# 策略梯度学习笔记

策略梯度（Policy Gradient）与 DQN 家族最大的区别：不再"学 Q 值再 argmax"，而是**直接学策略函数 $\pi_\theta(a|s)$**——这是从"价值学习"到"策略学习"的范式转变，也是打通连续动作控制的第一步。

> 配套路线：REINFORCE → Actor-Critic (A2C) → PPO →（控制方向补充：MPC 与 RL 结合）

## 目录

| 章节 | 内容 |
|---|---|
| 一 | REINFORCE（原理 + 推导 + 改进：Baseline / 熵正则 / 回报标准化） |
| 二 | Rainbow A2C（ GAE / 并行环境 / 共享网络（与Dueling DQN区别） / 奖励重塑 |
| 三 | PPO（重要性采样 / clip 信任区域 / 多 epoch 复用） |
| 四 | 控制方向补充：MPC 与 RL 结合（后续补充） |

> 本笔记当前包含**章节一：REINFORCE、章节二：Rainbow A2C、章节三：PPO**；后续章节随学习进度逐步补充。

## 一、REINFORCE

### 1.1 为什么从"学Q值再argmax"转向"直接学策略"？

#### 1.1.1 先回忆 DQN 的决策方式

DQN 学的是一个价值函数 $Q(s,a;\theta)$，决策时对所有动作取最大：

$$\pi(s) = \arg\max_a Q(s,a;\theta)$$

这套"先学价值、再取最大"的范式有两个结构性局限。

#### 1.1.2 局限一：连续动作空间下 argmax 不可行

$\arg\max_a$ 意味着要把**所有**候选动作的 $Q$ 值都算一遍才能选。动作是离散的（左/右）可以枚举；但动作一旦连续（如电机力矩取 $[u_{\min}, u_{\max}]$ 内任意值），候选动作有无穷多个，**无法遍历**。这是 DQN 家族无法直接做连续控制的最根本原因。

#### 1.1.3 局限二：决策是确定性的，表达不了随机策略

$\arg\max$ 永远只输出一个动作，等价于"以概率 1 选最优动作"。但很多场景需要**随机策略**：探索阶段需要"偶尔乱走"；博弈类问题中随机策略本身更优（如石头剪刀布）。DQN 的随机性只能靠外挂的 $\varepsilon$-贪婪，而不是由策略本身表达。

#### 1.1.4 策略梯度：直接学"概率"

策略梯度方法不再学 $Q$ 值，而是**直接把策略本身参数化**。设神经网络对状态 $s$ 输出一组"打分"（logits）$z(s) = f_\theta(s)$，用 softmax 变成概率分布：

$$\pi_\theta(a|s) = \frac{e^{z_a(s)}}{\sum_{a'} e^{z_{a'}(s)}}$$

这就是**随机策略**：决策时按这个分布采样，而不是取最大值。

| | DQN 的 Q 网络 | 策略网络 |
|---|---|---|
| 输出 | 各动作的 Q 值 | 各动作的概率（softmax 后） |
| 决策 | $\arg\max$（确定性） | 按概率采样（随机） |
| 连续动作 | 不可行（无法 argmax） | 可行（高斯策略，见 PPO 阶段） |

> **一句话**：DQN 学"这个动作值多少钱"，策略梯度学"这个状态该怎样随机地选动作"；前者靠 argmax 决策，后者靠采样决策——这就是两者最本质的分水岭。

### 1.2 策略梯度定理——目标函数的梯度从哪来？

#### 1.2.1 目标函数：显式定义"表现好坏"

学习 $Q$ 值时目标藏在贝尔曼方程里（让 $Q$ 逼近最优）；策略梯度需要一个**显式**的目标函数：

$$J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}\Big[\underbrace{\sum_{t=0}^{T}\gamma^t r_t}_{R(\tau)}\Big]$$

其中 $\tau = (s_0,a_0,r_0,s_1,a_1,\dots)$ 是一条轨迹（trajectory），$R(\tau)$ 是整条轨迹的折扣回报。学习任务变成最优化问题：

$$\theta^* = \arg\max_\theta J(\theta)$$

做法是对 $J(\theta)$ 做**梯度上升**。唯一的问题是：梯度怎么算？——期望里的概率分布本身依赖 $\theta$，不能直接求导。这就是策略梯度定理要解决的问题。

#### 1.2.2 轨迹概率：策略 × 模型

一条轨迹出现的概率由两部分决定：

$$P(\tau|\theta) = \rho(s_0)\prod_{t=0}^{T}\pi_\theta(a_t|s_t)\,P(s_{t+1}|s_t,a_t)$$

- $\rho(s_0)$：初始状态分布；
- $\pi_\theta(a_t|s_t)$：**策略**（我们能控制的）；
- $P(s_{t+1}|s_t,a_t)$：**环境转移概率 / 模型**（我们控制不了的）。

把期望写成积分，$$ J(\theta) = \int R(\tau)p(\tau \mid \theta)d\tau $$


注意：它是一个 (2T+1) 重积分（假设状态和动作都是一维；如果是多维向量，每一维都还要再展开，积分重数就更高）

假设：

- 状态s，一维实数
- 动作a，一维实数
- 轨迹只有 2 步：s0,a0,s1

积分形式是：


$$J = \iiint R(s_0, a_0, s_1) p(s_0, a_0, s_1) \, ds_0 \, da_0 \, ds_1$$


概率密度可以按因果关系分解：

$$
p(s_0, a_0, s_1) = \rho_0(s_0) \cdot \pi_\theta(a_0|s_0) \cdot P(s_1|s_0, a_0)
$$

代入后：

$$
J = \int_{s_0} \rho_0(s_0) \left[ \int_{a_0} \pi_\theta(a_0|s_0) \left( \int_{s_1} P(s_1|s_0, a_0) R(s_0, a_0, s_1) \, ds_1 \right) da_0 \right] \, ds_0
$$

但是实际上不会这样求解。



再对 $\theta$ 求导（积分与求导可交换）：

$$\nabla_\theta J(\theta) = \int_\tau \nabla_\theta P(\tau|\theta)\,R(\tau)\,d\tau$$

**困难**：$P(\tau|\theta)$ 是一长串概率的乘积。对乘积求导，乘法法则要展开成天文数字的项，无法处理。

#### 1.2.3 log 求导技巧：把"乘积"变成"求和"

**注意：在机器学习中，log默认的底数都是e**

微积分里有恒等式：

$$\frac{d}{dx}\log f(x) = \frac{f'(x)}{f(x)} \;\Longrightarrow\; f'(x) = f(x)\cdot\frac{d}{dx}\log f(x)$$

把它用在 $P(\tau|\theta)$ 上：

$$\nabla_\theta P(\tau|\theta) = P(\tau|\theta)\cdot\nabla_\theta \log P(\tau|\theta)$$

**这一步的价值**：$\log$ 把"乘积"变成"求和"，对"和"求导可以逐项进行——乘法法则的灾难被 $\log$ 化解了。

#### 1.2.4 模型项消失：不知道环境模型也能算梯度

把轨迹概率取对数：

$$\log P(\tau|\theta) = \underbrace{\log \rho(s_0)}_{\text{与 }\theta\text{ 无关}} + \sum_{t=0}^{T}\log \pi_\theta(a_t|s_t) + \underbrace{\sum_{t=0}^{T}\log P(s_{t+1}|s_t,a_t)}_{\text{与 }\theta\text{ 无关}}$$

对 $\theta$ 求导，**初始分布和环境模型都不含 $\theta$，梯度为 0**：

$$\nabla_\theta \log P(\tau|\theta) = \sum_{t=0}^{T}\nabla_\theta \log \pi_\theta(a_t|s_t)$$

> ⭐ **这是策略梯度最深刻的一点**：环境模型项在求导后**消失**了。所以策略梯度天然是 model-free——不需要动力学模型，只要能从真实环境采样轨迹即可。对比你毕设的 MPC：MPC 必须有模型，而策略梯度不需要。

#### 1.2.5 策略梯度定理

把 1.2.3、1.2.4 代回 1.2.2：

$$\nabla_\theta J(\theta) = \int_\tau P(\tau|\theta)\left[\sum_{t=0}^{T}\nabla_\theta \log \pi_\theta(a_t|s_t)\right]R(\tau)\,d\tau = \mathbb{E}_{\tau\sim\pi_\theta}\left[\sum_{t=0}^{T}\nabla_\theta \log \pi_\theta(a_t|s_t)\cdot R(\tau)\right]$$

写成期望形式即**策略梯度定理**：

$$\boxed{\nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}\left[\sum_{t=0}^{T}\nabla_\theta \log \pi_\theta(a_t|s_t)\cdot R(\tau)\right]}$$

#### 1.2.6 蒙特卡洛估计：用一条轨迹近似期望

期望没法精确算（要遍历所有轨迹），但可以**采样**——这与你学过的 MC 控制"用一条轨迹估计 Q 值"是同一个思想：

$$\nabla_\theta J(\theta) \approx \sum_{t=0}^{T}\nabla_\theta \log \pi_\theta(a_t|s_t)\cdot R(\tau)$$

采样的轨迹越多、估计越准。**REINFORCE = 策略梯度定理 + 蒙特卡洛采样**。

### 1.3 为什么可以用 $G_t$ 替换整条轨迹回报 $R(\tau)$？

#### 1.3.1 因果直觉

定理里每一项都乘着整条轨迹的回报 $R(\tau)=\sum_{k=0}^{T}\gamma^k r_k$。但直觉上，**t 时刻的动作 $a_t$ 只影响 t 时刻之后的回报**——过去的奖励已经发生，与 $a_t$ 无关。

于是把 $R(\tau)$ 换成"从 t 时刻往后的折扣回报"：

$$G_t = \sum_{k=t}^{T}\gamma^{k-t}r_k = r_t + \gamma r_{t+1} + \gamma^2 r_{t+2} + \cdots$$

#### 1.3.2 为什么期望不变

严格的论证是：$R(\tau)$ 中"t 时刻之前"的那些项与 $a_t$ 无关，代入梯度后会在期望中互相抵消。因此：

$$\mathbb{E}_\tau\left[\nabla_\theta \log \pi_\theta(a_t|s_t)\cdot R(\tau)\right] = \mathbb{E}_\tau\left[\nabla_\theta \log \pi_\theta(a_t|s_t)\cdot G_t\right]$$

证明：

把总回报拆成过去和未来两部分(前面是过去回报，后面是未来回报)：

$$
R(\tau) = \sum_{k=0}^{t-1} r_k + \sum_{k=t}^{T} r_k
$$


对于时刻t的动作，它的梯度更新项是：

$$
R(\tau) \nabla_\theta \log \pi_\theta(a_t | s_t) = \text{过去回报} \cdot \nabla_\theta \log \pi + \text{未来回报} \cdot \nabla_\theta \log \pi
$$

我们来看 **过去回报** 那一项在期望下的表现：

$$
\mathbb{E}_{a_t \sim \pi}[\text{过去回报} \cdot \nabla_\theta \log \pi(a_t | s_t)]
$$

由于过去回报是在a_t被采样之前已经确定的，它与 a_t的采样是独立的。因此可以分离：

$$
= \text{过去回报} \cdot \mathbb{E}_{a_t \sim \pi}[\nabla_\theta \log \pi(a_t | s_t)]
$$

而 $$\mathbb{E}_{a_t \sim \pi}[\nabla_\theta \log \pi(a_{t+1} | s_t)] = 0$$(这是 log 概率梯度的基本性质)。

所以 **过去回报** 对整个梯度期望的贡献为零。

最终 REINFORCE 实际使用的形式：

$$\boxed{\nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}\left[\sum_{t=0}^{T}\nabla_\theta \log \pi_\theta(a_t|s_t)\cdot G_t\right]}$$

理解：假设有一条轨迹:s0,a0,s1,a1,s2......
那么实际使用的损失函数就是s0选取a0的对数概率乘s0出发到轨迹结束的G0+s1选取a1的对数概率乘s1出发到轨迹结束的G1+......


#### 1.3.3 反向迭代计算 $G_t$

一回合的奖励序列 $[r_0, r_1, \dots, r_T]$ 已知后，从末尾往前推：

$$G_T = r_T, \qquad G_t = r_t + \gamma\,G_{t+1}$$

一次遍历即可得到所有时刻的 $G_t$——和你在 MC 控制里反向算 $G$ 完全一样。

> **一句话**：$G_t$ 是"这个动作之后带来的回报"，用它给每个动作打分，既符合因果直觉，又不改变梯度期望。

### 1.4 REINFORCE 算法：蒙特卡洛策略梯度

#### 1.4.1 完整流程（无基线版）

REINFORCE 就是"策略梯度定理 + 蒙特卡洛采样"，流程：

```
1. 用当前策略 π_θ 与环境交互，采样一整条轨迹
   （每一步记录 log π_θ(a_t|s_t) 和奖励 r_t）
2. 回合结束 → 反向迭代算出每个时刻的 G_t
3. 构造损失（负号：优化器只做最小化，而我们要最大化 J(θ)）：
   loss = -Σ_t log π_θ(a_t|s_t) · G_t
4. 对 θ 做一步梯度下降：
   θ ← θ + α Σ_t ∇_θ log π_θ(a_t|s_t) · G_t
5. 这条轨迹用完即弃，进入下一个回合
```

写成更新公式：

$$\theta \leftarrow \theta + \alpha\sum_{t=0}^{T}\underbrace{G_t}_{\text{评估}}\cdot\underbrace{\nabla_\theta\log\pi_\theta(a_t|s_t)}_{\text{方向}}$$

#### 1.4.2 直觉：适者生存

- $G_t > 0$：这个动作带来了好结果 → 增大 $\log\pi_\theta(a_t|s_t)$ → 下次在 $s_t$ **更可能**选 $a_t$；
- $G_t < 0$：坏结果 → 降低该动作概率。

整体就是一句话：**"回报高的轨迹上的动作被加强，回报低的被削弱"**——像进化论里的适者生存。

#### 1.4.3 为什么 REINFORCE 是 on-policy？为什么经验回放会失效？

这是策略梯度与 DQN 最深刻的工程区别：

- DQN 学的是 $Q$ 值。$Q$ 值描述"状态-动作对值多少钱"，与数据是**哪条策略**采的无关 → off-policy，旧数据可以反复用（经验回放）。
- REINFORCE 学的是**策略本身**。损失里 $\log\pi_\theta(a_t|s_t)$ 必须是**产生这条轨迹的那份 $\theta$**——因为梯度推导里"轨迹概率"就是由当前 $\theta$ 决定的。用旧策略采的轨迹更新新策略，$\log$ 概率对不上，期望就错了。

所以策略梯度方法**每条轨迹用完即弃**，样本效率低，这是它相对 DQN 的工程代价。

> **一句话**：REINFORCE 的梯度公式里每一项都带着"当前策略的对数概率"，所以数据必须由当前策略现采现用——经验回放在 on-policy 方法里天然失效。

### 1.5 改进一：Baseline 减方差

#### 1.5.1 问题：REINFORCE 方差大

$G_t$ 是一整条轨迹的回报，包含大量环境随机性。即使策略很好，一条"倒霉"的轨迹也可能 $G_t$ 很低，导致好动作被错误地削弱 → 训练震荡、收敛慢。

#### 1.5.2 改进：减去一个与动作无关的基线

原始策略梯度估计：

$$
\nabla_\theta J(\theta) = \mathbb{E}_\tau \left[ \sum_t G_t \nabla_\theta \log \pi_\theta(a_t | s_t) \right]
$$

带基线$b(s_t)$后：

$$
\nabla_\theta J(\theta) = \mathbb{E}_\tau \left[ \sum_t (G_t - b(s_t)) \nabla_\theta \log \pi_\theta(a_t | s_t) \right]
$$

我们只增加了这一项：

$$
\mathbb{E}_\tau \left[ \sum_t b(s_t) \nabla_\theta \log \pi_\theta(a_t | s_t) \right]
$$

如果这项期望为零，则带基线不改变梯度期望。

对于任意状态$s$，动作$a$ 从策略$\pi_\theta(\cdot|s)$中采样，有：

$$
\mathbb{E}_{a \sim \pi_\theta(\cdot|s)}[\nabla_\theta \log \pi_\theta(a|s)] = 0
$$

证明：

$$
\mathbb{E}_{a \sim \pi}[\nabla_\theta \log \pi(a|s)] = \sum_a \pi(a|s) \nabla_\theta \log \pi(a|s)
$$

$$
= \sum_a \pi(a|s) \frac{\nabla_\theta \pi(a|s)}{\pi(a|s)}
$$

$$
= \sum_a \nabla_\theta \pi(a|s)
$$

$$
= \nabla_\theta \sum_a \pi(a|s)
$$

$$
= \nabla_\theta 1
$$

$$
= 0
$$

>① 第一个等式：期望的定义

$$
\mathbb{E}_{a \sim \pi}[\nabla_\theta \log \pi(a|s)] = \sum_a \pi(a|s) \nabla_\theta \log \pi(a|s)
$$

这是离散随机变量期望的定义。

- 对于离散随机变量 $X$，其概率质量函数为 $p(x)$，则函数 $f(X)$ 的期望是：

$$
\mathbb{E}[f(X)] = \sum_x p(x)f(x)
$$

- 在这里，随机变量是动作 $a$，其概率分布是策略 $\pi(a|s)$（给定状态 $s$ 下的条件概率）。

- 我们要对 $f(a) = \nabla_\theta \log \pi(a|s)$ 求期望，所以：

$$
\mathbb{E}_{a \sim \pi}[\nabla_\theta \log \pi(a|s)] = \sum_a \pi(a|s) \cdot \nabla_\theta \log \pi(a|s)
$$

如果动作空间是连续的，求和换成积分：

$$
\mathbb{E}[\cdot] = \int \pi(a|s) \cdot \nabla_\theta \log \pi(a|s) da
$$
>② 第二个等式：对数导数的链式法则

$$
\nabla_\theta \log \pi(a|s) = \frac{\nabla_\theta \pi(a|s)}{\pi(a|s)}
$$

这是微积分中的对数求导法则。

- 对于一元函数：  
  $$
  \frac{d}{dx} \log f(x) = \frac{f'(x)}{f(x)}
  $$

- 对于多元函数（参数向量 $\theta$），同样有：

  $$
  \nabla_\theta \log \pi(a|s) = \frac{1}{\pi(a|s)} \nabla_\theta \pi(a|s)
  $$

- 因为 $\log$ 函数对 $\pi$ 的导数是 $1/\pi$，再乘以 $\pi$ 对 $\theta$ 的梯度。
其中倒数第二步使用了概率分布归一化条件$ \sum_a \pi(a|s) = 1$。因为所有动作的概率总和恒为 1，因此提高某个动作的概率必然同时降低其他动作的概率，所以概率梯度的加权平均为零。

现在看增加的那一项：

$$
\mathbb{E}_\tau \left[ \sum_t b(s_t) \nabla_\theta \log \pi_\theta(a_t | s_t) \right] = \sum_t \mathbb{E}_\tau [b(s_t) \nabla_\theta \log \pi_\theta(a_t | s_t)]
$$

利用条件期望：

$$
\mathbb{E}_\tau[\cdot] = \mathbb{E}_{s_t} \left[ \mathbb{E}_{a_t \sim \pi(\cdot | s_t)}[\cdot | s_t] \right]
$$

所以：

$$
\mathbb{E}_\tau[b(s_t) \nabla_\theta \log \pi_\theta(a_t | s_t)] = \mathbb{E}_{s_t} \left[ b(s_t) \mathbb{E}_{a_t \sim \pi(\cdot | s_t)}[\nabla_\theta \log \pi_\theta(a_t | s_t) | s_t] \right]
$$

$$
= \mathbb{E}_{s_t} [b(s_t) \cdot 0]
$$

$$
= 0
$$

**关键：** $b(s_t)$ 只依赖于状态，不依赖于动作，所以可以提到对动作的期望之外；而内层的动作期望 $\mathbb{E}_{a_t}[\nabla_\theta \log \pi_\theta(a_t | s_t)] = 0$，于是整个项为零。

**对于上述推导中，条件期望的公式**：

**条件期望迭代法则（Tower Rule）**
告诉我们：对于任意两个随机变量 $X, Y$，有：

$$
\mathbb{E}_{X,Y} [f(X,Y)] = \mathbb{E}_X [\mathbb{E}_{Y|X} [f(X,Y)|X]]
$$

也就是说，可以先固定 $X$，对 $Y$ 的条件分布求期望，然后再对 $X$ 求期望。

本质就是全期望法则：E(X) = E(E(X|Y))，就是对X的期望等于先求在Y的条件下X的条件期望，再求Y的期望。

**应用到我们的情况**

在我们的函数中，$f(s_t, a_t) = b(s_t) \nabla_\theta \log \pi_\theta(a_t | s_t)$。

注意：

- $s_t$ 是状态，在给定 $s_t$ 后，动作 $a_t$ 从条件分布 $\pi_\theta(\cdot | s_t)$ 中采样。
- 函数 $f$ 中，$b(s_t)$ 只依赖于 $s_t$，而 $\nabla_\theta \log \pi_\theta(a_t | s_t)$ 依赖于 $a_t$（也依赖于 $s_t$ 因为策略条件在 $s_t$）。

因此，我们可以先对 $a_t$ 求条件期望（固定 $s_t$），再对 $s_t$ 求期望：

$$
\mathbb{E}_\tau [b(s_t) \nabla_\theta \log \pi_\theta(a_t | s_t)] = \mathbb{E}_{s_t} [\mathbb{E}_{a_t \sim \pi_\theta(\cdot | s_t)} [b(s_t) \nabla_\theta \log \pi_\theta(a_t | s_t) | s_t]]
$$

内部条件期望中，给定 $s_t$，则 $b(s_t)$ 是常数，可以提出：

$$
\mathbb{E}_{a_t \sim \pi_\theta(\cdot | s_t)} [b(s_t) \nabla_\theta \log \pi_\theta(a_t | s_t) | s_t] = b(s_t) \mathbb{E}_{a_t \sim \pi_\theta(\cdot | s_t)} [\nabla_\theta \log \pi_\theta(a_t | s_t) | s_t]
$$

而根据 score function 的性质：

$$
\mathbb{E}_{a_t \sim \pi_\theta(\cdot | s_t)} [\nabla_\theta \log \pi_\theta(a_t | s_t) | s_t] = 0
$$

所以内部期望为零，最终整个期望为零。因此减去基线后不改变梯度期望。

**直觉理解：策略梯度更新本质上是在调整动作概率：让回报高于基线的动作概率增大，低于基线的动作概率减小。基线只提供了一个“比较基准”，它像对所有动作的 log 概率导数施加了一个相同的偏移量。由于动作概率导数在所有动作上的平均值为零，这个均匀偏移不会影响整体的更新方向。因此，减去的基线的净效果为零，却能在单次采样中降低方差（因为样本估计中，基线项不是严格零，但它的波动与回报项部分抵消）。**


#### 1.5.4 为什么方差会降低？

$G_t$ 围绕其均值波动，$G_t - b(s_t)$ 相当于把波动中心移到 0 附近。直观理解：**原来"好动作 vs 坏动作"的区分信号淹没在大的整体回报波动里；减去基线后，波动中心归零，每个动作的相对好坏更清晰**。数学上可以证明，合适的 $b(s_t)$ 能显著降低梯度的方差。

> **结论：基线 $b(s_t)$ 只要与动作 $a_t$ 无关，就不会改变梯度的期望，却能显著降低方差——这是一次"免费"的改进。与Dueling DQN的改进思想类似，都是考虑相对平均值的偏差**

### 1.6 改进二：熵正则（Entropy Regularization）

#### 1.6.1 问题：策略坍缩

在奖励恒正的环境（如 CartPole 每步 +1）里，所有 $G_t > 0$，REINFORCE 会"所有动作都被加强"，策略概率逐渐坍缩到一个动作（探索消失）。对于离散动作的 softmax 策略，增加某个动作的对数概率，会导致其他动作的概率被动降低（因为概率总和恒为 1）。
因此，只要某个动作一开始因为随机性获得了较高的回报，策略就会持续增强该动作的概率，同时压制其他动作。智能体（或大模型）的动作概率分布过早地变得极其单一、极端化，导致探索能力彻底丧失、输出高度同质化的退化现象。高度同质化的退化现象。简单来说，就是模型在强化学习过程中“学傻了”或“走火入魔”了，只认准某一种或某几种固定的套路，再也无法生成多样化的结果。一旦坍缩到的不是好动作，性能就会崩——无基线 REINFORCE 常出现的"先涨后崩"就是这个原因。

#### 1.6.2 改进：在目标里加一项熵

定义策略在状态 $s$ 下的熵（衡量随机程度）：

$$H(\pi_\theta(\cdot|s)) = -\sum_a \pi_\theta(a|s)\log\pi_\theta(a|s)$$

#### 1. 熵的公式：为什么是 $-\sum \pi \log \pi$？

你提到的“p乘以对数概率的和”实际上还差一个**负号**。标准的熵定义是：

$$
H(\pi) = -\sum_a \pi(a|s) \log \pi(a|s)
$$

这个公式来源于**信息论**，表示随机变量的不确定性或平均信息量。

#### ① 单个事件的信息量：$-\log p(a)$

- 如果一个事件发生的概率 $p$ 很小，那它一旦发生，我们会感到“惊讶”，它携带的信息量很大。
- 例如，太阳从西边升起（概率几乎为 0）会带来巨大的信息量。
- 相反，概率接近 1 的事件（如太阳从东边升起）发生后，我们并不惊讶，信息量接近 0。

信息论中定义事件 $a$ 的信息量为：

$$
I(a) = -\log p(a)
$$

因为 $p(a) \leq 1$，所以 $\log p(a) \leq 0$，因此 $-\log p(a) \geq 0$，信息量是非负的。概率越小，信息量越大。

#### ② 熵是信息量的期望

熵 $H$ 是所有可能事件的信息量的 **平均值**（期望）。期望的计算方法就是：每个事件的信息量乘以它发生的概率，再求和：

$$
H = \mathbb{E}_{a \sim \pi}[I(a)] = \sum_a p(a) \cdot I(a) = \sum_a p(a) \cdot (-\log p(a)) = -\sum_a p(a) \log p(a)
$$

所以：

- 每一项是 $p(a) \cdot (-\log p(a))$，即“概率 × 信息量”。
- 求和后就是平均不确定性。

这就是为什么熵是“概率乘对数概率的和”再加负号。

熵越大 → 策略越随机；熵 = 0 → 策略退化为确定性（坍缩）。把熵加入目标函数：

$$\tilde{J}(\theta) = J(\theta) + \beta\,\mathbb{E}_s\big[H(\pi_\theta(\cdot|s))\big]$$

$\beta > 0$ 是熵系数。最大化 $\tilde J$ = 既要回报高、又要策略别太确定 → 鼓励探索、防止坍缩。

#### 1.6.3 损失怎么写

对应到损失函数（最小化）：

$$\text{loss} = -\sum_t \log\pi_\theta(a_t|s_t)\,A_t - \beta\sum_t H(\pi_\theta(\cdot|s_t))$$

代码里用 PyTorch 一行：`dist.entropy()`（Categorical 分布自带）。

> **类比 DQN 家族**：熵正则在策略梯度里的角色 ≈ NoisyNet / ε-贪婪在 DQN 里的角色——都是"探索机制"，但熵正则更优雅：它由策略本身表达随机性，不需要外挂探索。

> **一句话**：熵正则用"鼓励随机"对抗"策略坍缩"，和 baseline 正交，可自由组合。

### 1.7 改进三：回报标准化（Reward Normalization）

#### 1.7.1 问题：$G_t$ 的尺度不稳定

不同回合的 $G_t$ 量级差异很大（一条短轨迹 $G_t$ 很小，一条长轨迹 $G_t$ 很大）。$G_t$ 直接乘进梯度里，会导致：
- 长轨迹的样本把梯度"带飞"，短轨迹的样本几乎不起作用；
- 更新步长随回报尺度忽大忽小，训练不稳定。

#### 1.7.2 改进：把整条轨迹的 $G_t$ 标准化

对本回合的 $G_t$ 做"减均值、除标准差"：

$$A_t = \frac{G_t - \bar{G}}{\text{std}(G) + \varepsilon}$$

- 减均值 $\bar G$：这本身就是一种**动态基线**（baseline，1.5 节）——把评估中心移到 0；
- 除标准差：把梯度尺度归一化，让每个回合对更新的贡献大致相当，$\varepsilon$是小常数，防止除以0。

#### 1.7.3 为什么有效

- 标准化后 $A_t$ 有大致固定的量纲 → 学习率 $\alpha$ 不需要针对回报尺度反复调；
- 每个回合的"相对好坏"（$A_t > 0$ 或 $<0$）成为主要信号，而不是绝对大小。

> **一句话**：回报标准化 = "动态基线 + 归一化"，本质是让每个回合的评估信号可比较，工程上极常用（后续 PPO 的优势归一化也是这个思路）。

### 1.8 伏笔：从 $G_t$ 到 $V(s)$——A2C 的 critic 雏形

#### 1.8.1 $G_t$ 仍然太"脏"

减了基线后方差小了，但 $G_t$ 仍是一整条轨迹的随机和。理想情况下，我们想评估的是"**在状态 $s_t$ 下选 $a_t$，比平均好多少**"，即优势：

$$A_t = G_t - V(s_t)$$

其中 $V(s_t)$ 是状态价值——"从 $s_t$ 出发能拿到的期望回报"。

#### 1.8.2 用神经网络学 $V(s)$：critic 诞生

$V(s)$ 本身可以用一个神经网络去学（让 $V_\phi(s)$ 去逼近 $G_t$，本质是 MC/TD 价值学习）。这个网络就是 **critic**（评论家），负责给 actor（策略网络）打分数：

- **actor**（策略网络 $\pi_\theta$）：负责"怎么做"——输出动作概率；
- **critic**（价值网络 $V_\phi$）：负责"做得怎么样"——输出状态价值作为基线。

这就是下一阶段 **Actor-Critic (A2C)**：actor 用 $A_t = G_t - V_\phi(s_t)$ 当评估器更新策略，critic 自己也在学习。**REINFORCE with Baseline 里的基线，正是 A2C 中 critic 的雏形。**

#### 1.8.3 对照 DQN：目标网络在 AC 里的角色

| | DQN | Actor-Critic (A2C) |
|---|---|---|
| 价值网络 | Q 网络 + 目标网络（压低自举偏差） | critic $V_\phi(s)$（作为基线，降低方差） |
| 策略网络 | 无（隐式 argmax） | actor $\pi_\theta(a\|s)$（显式策略） |
| 目标网络角色 | 提供稳定的 $\max Q(s',a')$ | 暂无对应物；A2C 引入自举后才会出现类似问题 |

### 1.9 REINFORCE 与 DQN 本质区别总结

| 维度 | DQN 家族 | REINFORCE |
|---|---|---|
| 学什么 | $Q(s,a)$ 价值函数 | $\pi_\theta(a\|s)$ 策略本身 |
| 怎么决策 | $\arg\max_a$（确定性） | 从 $\pi_\theta$ 采样（随机） |
| 连续动作 | ❌ 无法 argmax | ✅ 高斯策略可行 |
| 数据利用 | off-policy + 经验回放 | on-policy，轨迹用完即弃 |
| 目标网络 | 有（稳定目标值） | 无 |
| 梯度来源 | 贝尔曼误差（TD） | 整条轨迹 $G_t$（MC） |
| 方差 | 较小 | 很大 → 需要 baseline / critic |

> **一句话总结**：REINFORCE 用"采样一条轨迹 → 拿 $G_t$ 给每个动作打分 → 梯度上升"来直接优化策略；它天然支持连续动作且不需要环境模型，代价是 on-policy 带来的样本浪费和高方差——这两大缺点正是通往 A2C（用 critic 减方差）和 PPO（稳定更新）的道路。

## 二、Rainbow A2C（Actor-Critic）

### 2.1 动机：REINFORCE 剩哪两个毛病？

学完 REINFORCE Rainbow（基线 + 熵正则 + 回报标准化），方差大和探索退化已缓解，但框架仍有**两个结构性毛病**：

| 毛病 | 表现 |
|---|---|
| ① MC 回报方差仍大 | $G_t$ 要等**整个回合走完**才知道，且是一整条轨迹随机性的总和——baseline 只是"减轻"，没根治 |
| ② 回合制更新，样本效率低 | 每回合**只更新一次**；DQN 每步都能学，REINFORCE 只能等回合结束 |

A2C 的思路一句话：**请一个"评论家"（critic）实时打分，让策略（actor）每步都能学，且打分方差更小。**

### 2.2 核心结构：actor + critic 两个网络

| 角色 | 网络 | 负责什么 | 类比 |
|---|---|---|---|
| Actor（演员） | $\pi_\theta(a\|s)$ | **怎么做**——输出动作概率 | REINFORCE 的策略网络（已有） |
| Critic（评论家） | $V_\phi(s)$ | **做得怎么样**——输出状态价值 | REINFORCE 里 baseline 的"升级版" |

**Critic 就是 1.8 节伏笔里的 $V(s)$ 网络**——它取代"本回合回报均值"这种粗糙统计基线，变成能**随状态变化、每步都更新**的精准基线。

### 2.3 核心推导一：用 TD 目标替代 MC 回报

#### 2.3.1 先看 REINFORCE 的优势

你在 REINFORCE 里用的（笔记 1.5）：

$$A_t = G_t - b, \qquad G_t = \sum_{k=t}^{T}\gamma^{k-t}r_k \quad(\text{MC，要等回合结束})$$

#### 2.3.2 用一步 TD 目标替换 $G_t$

**注意Gt，V,Q的区别**：

Gt指的是是从时间步$t$ 开始到游戏结束，实际采样到的所有折扣奖励的总和，它是真实的、基于样本的、带随机性的具体数值。

$Q(s, a)$是在状态$s$采取动作$a$之后，未来所有可能获得的$G_{t}$的数学期望（平均值）。

$V$指在状态 $s$ 时，沿着某个策略走下去，未来收益$G_{t}$的期望值（平均收成）。

但是实际上三者者在不同视角下常常互相替代，在蒙特卡洛方法中，常用跑完的一个轨迹真实回报Gt来更新Q/V。在TD中用r+γQ/V来替代Gt（即A2C的优势中的Gt）。

至于用V还是Q，V代表当前状态有多好，用于状态评估或结合模型（即状态转移概率*(r+γV(s'))。Q用于无模型，可以直接对比不同动作的差异

MC无偏但方差大，TD有偏但方差小

Critic 能**每走一步**就估计"从 $s_{t+1}$ 出发值多少"，于是：

$$G_t \approx r_t + \gamma V_\phi(s_{t+1})$$

优势就变成 **TD 误差**：

$$\boxed{A_t = r_t + \gamma V_\phi(s_{t+1}) - V_\phi(s_t)}$$

**为什么这是"优势"？** 拆开看含义：
- $V_\phi(s_t)$：从 $s_t$ 出发的期望回报（= "平均水平"）——它就是基线；
- $r_t + \gamma V_\phi(s_{t+1})$：执行动作 $a_t$ 之后**实际走一步**的回报 + 对未来价值的估计；
- 两者之差 = "**选 $a_t$ 比平均水平好多少**"。$A_t>0$ 表示 $a_t$ 优于平均 → 加强。

#### 2.3.3 两个好处 + 一个代价

**好处**：
1. **不用等回合结束**——每走一步，用 $(s_t \to s_{t+1})$ 一步数据就能算 $A_t$，策略**每步都能更新**（样本效率↑）；
2. **方差小**——只依赖一步奖励和一步价值估计，而不是整条轨迹的随机和。

**代价**：引入了**自举（bootstrap）**——$V_\phi(s_{t+1})$ 是网络自己的估计，有偏。这就是"方差-偏差权衡"：REINFORCE 无偏但方差大，A2C 有偏但方差小。

> 📌 **关联 DQN**：$r_t + \gamma V(s_{t+1})$ 和 DQN 的 $r + \gamma\max Q(s',a')$ 是同一个东西——都是**贝尔曼方程的单步展开**。DQN 对 Q 自举，A2C 对 V 自举。

### 2.4 核心推导二：actor 更新（策略梯度定理直接套用）

Actor 的更新和 REINFORCE 完全同源（策略梯度定理，1.2 节）：

$$\nabla_\theta J(\theta) = \mathbb{E}\big[\nabla_\theta\log\pi_\theta(a_t|s_t)\cdot A_t\big]$$

损失（最小化）：

$$\text{loss}_{\text{actor}} = -\sum_t \log\pi_\theta(a_t|s_t)\cdot A_t$$

**公式没变，只换了 $A_t$**：从"MC 的 $G_t - b$"换成"TD 的 $r_t + \gamma V(s_{t+1}) - V(s_t)$"。直觉不变：$A_t > 0$ → 加强，$A_t < 0$ → 削弱；只是"平均"现在由 critic 精确刻画、每步实时算。

### 2.5 核心推导三：critic 怎么学（TD 学习）

Critic 的目标：让 $V_\phi(s_t)$ 逼近真实状态价值 $V^\pi(s_t)$。真实值未知，但贝尔曼方程给了递归关系：

$$V^\pi(s_t) = r_t + \gamma V^\pi(s_{t+1})$$

用"单步 TD 目标"当标签，让 $V_\phi(s_t)$ 去逼近它——**critic 的损失就是 TD 误差的平方**：

$$\text{loss}_{\text{critic}} = \frac{1}{2}\big(V_\phi(s_t) - (r_t + \gamma V_\phi(s_{t+1}))\big)^2$$

> 📌 **关联 DQN**：这个损失和 DQN 的 `loss = MSELoss(q_values, q_target)` **数学结构完全一样**——都是"让当前估计逼近单步贝尔曼目标"。区别：DQN 学 $Q(s,a)$（需要动作、要 argmax），A2C 的 critic 学 $V(s)$（不需要动作）。**你 DQN 里会的那套 TD 学习，搬到 critic 上就是现成的。**

### 2.6 核心推导四：GAE（广义优势估计）

#### 2.6.1 从 n-step 到 GAE

一步 TD 偏差偏大（$V(s_{t+1})$ 不准时误差直接进优势）。n-step 优势折中：

$$A_t^{(n)} = \left(\sum_{k=0}^{n-1}\gamma^k r_{t+k}\right) + \gamma^n V_\phi(s_{t+n}) - V_\phi(s_t)$$

关键观察：n-step 优势可以写成 **TD 误差的累加**。定义 TD 误差：

$$\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$$

则

$$A_t^{(1)} = \delta_t, \qquad A_t^{(2)} = \delta_t + \gamma\delta_{t+1}, \qquad A_t^{(n)} = \sum_{l=0}^{n-1}\gamma^l \delta_{t+l}$$

即n-step 优势就是未来 n 个 TD 残差的折扣和，具体证明省略，可以用归纳法证明。

#### 2.6.2 GAE：对"所有 n"做指数加权平均

GAE 把所有 n-step 优势加权平均（权重 $(1-\lambda)\lambda^{n-1}$，归一化后总和为 1）：

$$A_t^{\text{GAE}} = (1-\lambda)\sum_{n=1}^{\infty}\lambda^{n-1} A_t^{(n)}$$

关于权重：我们不想硬选一个 n，而是希望所有 n 的估计都参与，但越大的 n 权重越小（随着 n 增加，估计的方差越来越大，训练稳定性变差，宁愿有偏，也要控制方差）。一个自然的选择是几何衰减：1，$\lambda$，$\lambda^2$,$\lambda^3$......但是这些权重求和并非1，而是$1/(1-\lambda)$（等比数列求和）会把整个优势估计放大$1/(1-\lambda)$，因此每个权重还需要乘$1-\lambda$

**推导：为什么最后是 $(\gamma\lambda)^l$ 加权？**

把 $A_t^{(n)} = \sum_{l=0}^{n-1}\gamma^l\delta_{t+l}$ 代入并交换求和顺序：$\delta_{t+l}$ 出现在所有 $n \ge l+1$ 的项里，其系数为

$$(1-\lambda)\sum_{n=l+1}^{\infty}\lambda^{n-1}\gamma^l = (1-\lambda)\gamma^l\sum_{m=l}^{\infty}\lambda^{m} = (1-\lambda)\gamma^l\cdot\frac{\lambda^l}{1-\lambda} = (\gamma\lambda)^l$$

所以

$$\boxed{A_t^{\text{GAE}} = \sum_{l=0}^{\infty}(\gamma\lambda)^l\,\delta_{t+l}}$$

证明：

我们关注的是某个特定的 TD 残差，比如 $\delta_{t+l}$（即未来第 $l$ 步的残差）。

在 GAE 的定义中：

$$
A_t^{GAE} = (1 - \lambda) \sum_{n=1}^\infty \lambda^{n-1} A_t^{(n)}
$$

把 $n$ 步优势展开：

$$
A_t^{GAE} = (1 - \lambda) \sum_{n=1}^\infty \lambda^{n-1} \left( \sum_{k=0}^{n-1} \gamma^k \delta_{t+k} \right)
$$

**关键问题**: 对于固定的 $\delta_{t+l}$，它在哪些 $n$ 值对应的项里出现？

——只要满足 **内层求和的上限 $n-1 \geq l$**，即 $n \geq l+1$ 时，$\delta_{t+l}$ 就会出现。

（注意：当 $n \leq l$ 时，内层还没累加到这一项，所以不包含它。）

因此，提取 $\delta_{t+l}$ 的系数时，求和号 $n$ 必须从 $l+1$ 开始。

把 $\delta_{t+l}$ 的系数单独写出来：

$$
\text{Coeff}(\delta_{t+l}) = (1 - \lambda) \sum_{n=l+1}^{\infty} \lambda^{n-1} \cdot \gamma^l
$$

因为 $\gamma^l$ 与 $n$ 无关，直接提到前面：

$$
\text{Coeff} = (1 - \lambda) \gamma^l \sum_{n=l+1}^{\infty} \lambda^{n-1}
$$

令 $m = n - 1$。

当 $n = l + 1$ 时，$m = l$；

当 $n \to \infty$ 时，$m \to \infty$。

于是求和变为：

$$
\sum_{n=l+1}^\infty \lambda^{n-1} = \sum_{m=l}^\infty \lambda^m
$$


因为 $\lambda \in [0,1)$，这是一个从 $m = l$ 开始的无穷等比数列：

$$
\sum_{m=l}^\infty \lambda^m = \lambda^l + \lambda^{l+1} + \lambda^{l+2} + \cdots
$$

提取公因式 $\lambda^l$：

$$
= \lambda^l (1 + \lambda + \lambda^2 + \cdots) = \lambda^l \cdot \frac{1}{1 - \lambda}
$$


把结果代回系数表达式：

$$
\text{Coeff} = (1 - \lambda) \gamma^l \cdot \frac{\lambda^l}{1 - \lambda}
$$

分子分母的 $(1 - \lambda)$ 直接约掉，得到：

$$
\text{Coeff} = \gamma^l \lambda^l = (\gamma \lambda)^l
$$


- **直观含义**：这意味着 GAE 的最终形式可以写成单层求和：

$$
A_t^{GAE} = \sum_{l=0}^\infty (\gamma \lambda)^l \delta_{t+l}
$$

原来公式中，优势权重假设为abc....，对应的是123.....步的优势
正常的公式应该是a1+b2+c3+....
实际上就是a·1步的残差+b·2步的残差1+b·2步的残差2·残差2的折扣+c·3步的残差1+c·3步的残差2·折扣+c·3步的残差3·折扣

为了方便计算，就把各个残差的全部系数算出来加起来·第l个残差就是类似于合并每一步优势内的第l步TD误差系数。简化计算

#### 2.6.3 参数 $\lambda$ 的连续调节

- $\lambda = 0$：$A_t = \delta_t$（一步 TD，方差最小、偏差最大,注意不是0，第一项是0^0 = 1乘$\delta_t$）；
- $\lambda = 1$：$A_t = \sum_l \gamma^l\delta_{t+l} = G_t - V(s_t)$（MC 优势，无偏、方差最大，展开后消消乐，就是MC优势）；
- 常用 $\lambda \approx 0.95$。

**一个参数 $\lambda$ 连续调节方差-偏差权衡**，比固定整数 n-step 更平滑——这就是 PPO 里用的优势估计。

#### 2.6.4 代码计算：反向迭代

$$A_t = \delta_t + \gamma\lambda\,A_{t+1}$$

证明：

$$
A_t = \delta_t + \gamma \lambda A_{t+1}
$$

$$
= \delta_t + \gamma \lambda (\delta_{t+1} + \gamma \lambda A_{t+2})
$$

$$
= \delta_t + \gamma \lambda \delta_{t+1} + (\gamma \lambda)^2 A_{t+2}
$$

$$
= \delta_t + \gamma \lambda \delta_{t+1} + (\gamma\lambda)^2 \delta_{t+2} + \cdots
$$



从轨迹末尾往前推，一次遍历即可

实际代码都用这个递推，注意与MC/REINFORCE的倒序累加一致，就是把MC的r换成TD误差，G折扣因子换成gamma*lambda，外加一个中止标志。$A_t$就是每个时间步的优势（G-V）

### 2.7 并行环境（A2C 的灵魂）

同时开 N 个 CartPole（如 N=8），各自采样轨迹，收集满一批后**同步**更新一次。好处：
- 一个 batch 里同时有 N 条互不相关的轨迹 → 梯度估计更稳（天然数据多样性）；
- 这就是"同步版"（A2C）名字的由来（异步版是 A3C）。

实现上 gymnasium 提供 `gym.vector` 批量环境，reset/step 一次返回 N 个结果。细节在代码里看。

### 2.8 共享网络（简略，类比 Dueling）

actor 和 critic **共用底层特征提取层**，最后分两个输出头：

```text
输入 s → [共享层 128-128] → 头A: 动作概率 logits
                          → 头V: 状态价值 V(s)
```

> 思想和你学过的 **Dueling DQN** 一致：共享底层特征、减少参数、加速训练。区别只是 Dueling 分"价值/优势"两个头，A2C 分"actor/critic"两个头。

### 2.9 已学组件直接沿用（简略）

| 组件 | 来源 | 在 A2C 里 |
|---|---|---|
| 熵正则 | 笔记 1.6 | 照用：损失加 $-c_2 H(\pi(\cdot\|s))$，防坍缩 |
| 回报标准化 | 笔记 1.7 | 升级为"优势标准化"：对一批 $A_t$ 减均值除标准差 |
| 梯度裁剪 | 工程默认 | 照用 |

### 2.10 完整更新流程与总结

**一个更新步**（并行 N 个环境 × 滚动 T 步后）：

1. 用当前 actor 批量采样动作，环境批量 step；
2. critic 批量估计 $V(s)$；
3. 计算 GAE 优势（2.6）；
4. 优势标准化（2.9）；
5. 更新：

$$\text{loss} = \underbrace{-\sum_t\log\pi(a_t|s_t)A_t}_{\text{actor}} + c_1\underbrace{\big(V(s_t)-\text{target}\big)^2}_{\text{critic}} - c_2\underbrace{H(\pi(\cdot|s_t))}_{\text{熵正则}}$$

**与 REINFORCE 对照总结**：

| 维度 | REINFORCE (Rainbow) | Rainbow A2C |
|---|---|---|
| 网络 | 只有 actor | actor + critic（共享底层） |
| 评估信号 | $G_t$（MC，回合结束才知） | $A_t$（GAE，每步可算） |
| 更新时机 | 每回合一次 | 每步 / 每批一次 |
| 基线 | 统计量 | $V(s)$ 网络 |
| 方差/偏差 | 方差大、无偏 | 方差小、有偏（自举） |
| 数据 | on-policy，单环境 | on-policy，并行 N 环境 |
| 目标网络 | 无 | 通常无 |

> **一句话**：Rainbow A2C = "REINFORCE 的策略梯度公式 + DQN 的价值网络学习 + 并行环境 + GAE"。其中 critic 的 TD 学习、GAE、并行环境是新的；熵、优势标准化、共享底层是已学思想直接沿用。

### 2.11 补充：奖励重塑

**一个朴素但有效的奖励重塑方法**

```python
def shape_reward(states):
    """给"靠近着陆区、机身水平、下降放缓"加中间引导奖励，破解悬浮局部最优
    states: [N, 8]（LunarLander 观测：x, y, vx, vy, angle, angvel, leg1, leg2）
    返回: [N] 每个环境的引导奖励（用负距离 → 越接近目标值奖励越高）
    注意：只在【训练】时叠加；评估/测试用原始奖励，衡量真实表现
    """
    x, y, vx, vy, angle, angvel, leg1, leg2 = states.T   # 每行一个维度
    bonus = np.zeros(len(states))
    bonus -= np.abs(x) * 0.6   # 引导水平对准着陆区（观测 x=0 是着陆区中心，越接近 0 越好）
    bonus -= np.abs(angle)   * 0.3   # 引导机身水平（角度越接近 0 越好）
    bonus -= np.abs(vy)      * 0.5   # 引导垂直速度放缓（利于安全着陆）
    bonus -= np.abs(y)       *0.5
    return bonus  
    #.........
    #在循环体内部：#rewards = rewards + SHAPE_COEF * shape_reward(next_states)
    #就是给rewards一个惩罚，偏离目标越大，惩罚越大，迫使智能体朝目标
    #但是这个改变了原始奖励，设计不合理时，智能体倾向于满足重塑的奖励，而非满足最终奖励
``` 

**基于势函数的奖励重塑：potential-based shaping**
```python
def potential(states):
    """势函数 Φ(s)：越接近目标，势能越高"""
    x, y, vx, vy, angle, angvel, leg1, leg2 = states.T
    # 目标 x=0，角度=0，垂直速度=0
    return -1.0 * np.abs(x) - 1.0 * np.abs(angle) - 0.5 * np.abs(vy)
def potential_based_shaping(states, next_states, gamma=0.99):
    """基于势函数的奖励塑形：F = γΦ(s') - Φ(s)"""
    return gamma * potential(next_states) - potential(states)
    #......
    #循环体内部rewards = rewards + SHAPE_COEF * potential_based_shaping(states, next_states, GAMMA)
``` 

我们想给智能体一些额外的“即时反馈”，但又不想因此改变原始任务的最优解。

做法是：定义一个**状态势函数** $\Phi(s)$，表示状态 $s$ 的“好”程度。

然后定义附加奖励为：

$$
F(s, s') = \gamma \Phi(s') - \Phi(s)
$$

其中 $s'$ 是执行动作后的下一个状态，$\gamma$ 是折扣因子。

关于$\gamma \Phi(s')$的$\gamma$：

在折扣 MDP 中，我们最大化的是折扣累积回报：

$$
R_0 = r_0 + \gamma r_1 + \gamma^2 r_2 + \cdots
$$

如果我们给每一步都加上 shaping 奖励 $F(s_t, s_{t+1})$，那么总回报变成：

$$
R_0^{shaped} = \sum_{t=0}^\infty \gamma^t (r_t + F(s_t, s_{t+1}))
$$

我们希望附加的 shaping 项在总和上与路径无关，只依赖于初始状态和最终状态，这样就不会改变最优策略。

如果定义 $F(s, s') = \Phi(s') - \Phi(s)$（不乘 $\gamma$），那么累加后：

$$
\sum_{t=0}^\infty \gamma^t F(s_t, s_{t+1}) = \sum_{t=0}^\infty \gamma^t (\Phi(s_{t+1}) - \Phi(s_t))
$$

这个级数并不能简化为只依赖首尾的表达式，因为 $\gamma$ 的幂会改变每一项的权重。展开看：

$$
= \gamma^0 \Phi(s_1) - \gamma^0 \Phi(s_0) + \gamma^1 \Phi(s_2) - \gamma^1 \Phi(s_1) + \gamma^2 \Phi(s_3) - \gamma^2 \Phi(s_2) + \cdots
$$

中间项并不能完全抵消（因为相邻项前的 $\gamma$ 幂次不同），所以会影响最优策略。

如果定义 $F(s, s') = \gamma \Phi(s') - \Phi(s)$，则累加后：

$$
\sum_{t=0}^\infty \gamma^t (\gamma \Phi(s_{t+1}) - \Phi(s_t)) = \sum_{t=0}^\infty \gamma^{t+1} \Phi(s_{t+1}) - \sum_{t=0}^\infty \gamma^t \Phi(s_t)
$$

这两个求和正好错开一位，相减后中间所有项全部抵消，只剩下：

$$
= \lim_{T \to \infty} \gamma^{T+1} \Phi(s_{T+1}) - \Phi(s_0)
$$

在合理条件下（势能有界或折扣因子保证收敛），第一项趋近于 0，于是 shaping 的总贡献就是 $-\Phi(s_0)$，与路径无关，因此不改变最优策略。


### 2.12 补充：与Dueling DQN的区别

1. A2C 的 $A = G - V$（优势函数 Advantage）

- **全称**: Advantage Actor-Critic。

- **物理意义**: 这里的 $G$ 是实际采样到的回报（Return，即蒙特卡洛回报或多步累积回报），$V(s)$ 是状态价值函数。$A$ 表示“当前动作实际带来的效果，比预期的平均效果好多少”。

- **特点**:
  - $G$ 是随机采样得到的真实反馈（带有方差）。
  - $V(s)$ 是基准线（Baseline），代表在这个状态下所有动作的平均期望。
  - 它解决的是策略梯度中的高方差问题。

2. Dueling DQN 的 $A = Q - V$（优势函数）

- **全称**: Dueling Deep Q-Network。

- **物理意义**: 这里的 $Q$ 是对各个具体动作价值的估计，$V(s)$ 是该状态的平均价值。$A(s, a)$ 表示“选择某一个特定动作 $a$ 比该状态下平均水平好多少”。

- **特点**:
  - 这里的 $Q$ 和 $V$ 都是网络直接输出或计算出来的期望值，不是采样回报 $G$。
  - 它的设计目的是改变网络结构（分成 $V$ 流和 $A$ 流），让网络更容易学习哪些状态重要，而不必关心每个动作的具体回报。

- **二者V在概念定义上**：是一样的。  
  两者中的 $V(s)$ 都代表在状态 $s$ 下的期望回报（即在这个状态开始，未来能拿多少分）。

- **计算与实现上**：是不一样的。

  - **A2C 中的 $V(s)$**：由 Critic 网络单独输出，用来逼近真实回报 $G$ 的期望。它的训练目标是去拟合环境的实际收益。

  - **Dueling DQN 中的 $V(s)$**：它是 Q 网络内部结构的一种拆解。为了让 $V$ 和 $A$ 的定义可辨识（Identifiable），通常会强制约束优势函数的平均值或最大值为 0（例如  
    $$
    A(s, a) = \frac{1}{|A|} \sum_{a'} A(s, a')
    $$
    它不是一个单独去拟合外界真实回报的模块，而是辅助 $Q$ 值计算的中间产物。

Dueling DQN 里的 A 是网络结构内部对 Q 值的分解；A2C 里的 A 是训练时用回报和基线计算出来的优势估计。

## 三、PPO（Proximal Policy Optimization）

### 3.1 A2C 的两个"原罪"

在 LunarLander 上反复调参的经历说明，A2C 有两个核心痛点：

| 痛点 | 表现 |
|---|---|
| ① 步长难调 | LR 大一点就震荡，小一点就慢——每次更新没有任何机制限制"这步能走多远" |
| ② 样本只用一次 | 每收集一批数据只做一次梯度更新就丢掉（on-policy 严格性）→ 样本效率低 |

PPO 的思路：**① 让同一批数据能多用几次（解决样本效率）；② 同时保证多用几次也不会把策略推飞（解决步长）**。两个问题的答案同一个工具：**重要性采样**。

### 3.2 核心推导一：重要性采样——为什么旧数据能用多次？

#### 3.2.1 问题

目标函数依赖当前策略的期望：

$$J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}\big[R(\tau)\big]$$

但手上只有**旧策略 $\pi_{\theta_{\text{old}}}$** 采的数据（先采好、再更新），分布对不上。理论上的目标函数是当前策略的期望，但实际计算梯度时，用的确实是旧策略（采样时对应的策略）产生的数据。而每次更新后参数变动很小（参数更新步子小，学习率小），主要是为了保证“旧策略和新策略的差距很小”，从而让这种用旧数据近似新期望的做法在数学上近似成立。

#### 3.2.2 推导：把"对 $\pi_\theta$ 的期望"换成"对 $\pi_{old}$ 的期望"

把期望写成积分，乘一个"1"（$\frac{P_{old}}{P_{old}}$）：

$$J(\theta) = \int P_\theta(\tau)R(\tau)\,d\tau = \int P_{old}(\tau)\,\underbrace{\frac{P_\theta(\tau)}{P_{old}(\tau)}}_{\text{重要性权重}}\,R(\tau)\,d\tau = \mathbb{E}_{\tau\sim\pi_{old}}\Big[\frac{P_\theta(\tau)}{P_{old}(\tau)}R(\tau)\Big]$$

#### 3.2.3 关键化简：轨迹概率比 = 策略概率比

轨迹概率 $P_\theta(\tau) = \rho(s_0)\prod_t \pi_\theta(a_t|s_t)P(s_{t+1}|s_t,a_t)$（见 1.2 节）。做比值时，**初始分布和环境模型项全部约掉**：

$$\frac{P_\theta(\tau)}{P_{old}(\tau)} = \prod_t \frac{\pi_\theta(a_t|s_t)}{\pi_{old}(a_t|s_t)}$$

**原始策略梯度（无偏形式）**：

$$
\nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta} \left[ \sum_t \nabla_\theta \log \pi_\theta(a_t | s_t) \cdot \hat{A}_t \right]
$$

为了用旧策略 $\pi_{old}$ 的数据采样，我们引入重要性采样（只针对这一时刻的策略）：

$$
\nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_{old}} \left[ \sum_t \frac{\pi_\theta(a_t | s_t)}{\pi_{old}(a_t | s_t)} \nabla_\theta \log \pi_\theta(a_t | s_t) \cdot \hat{A}_t \right]
$$

**关键一步（构造原函数）**：

如果你对下面这个新目标函数 $L(\theta)$ 求梯度：

$$
L(\theta) = \mathbb{E}_{\tau \sim \pi_{old}} \left[ \sum_t \frac{\pi_\theta(a_t | s_t)}{\pi_{old}(a_t | s_t)} \hat{A}_t \right]
$$

因为 $\nabla_\theta \left( \frac{\pi_\theta}{\pi_{old}} \right) = \frac{\pi_\theta}{\pi_{old}} \cdot \nabla_\theta \log \pi_\theta$，所以：

$$
\nabla_\theta L(\theta) = \mathbb{E}_{\tau \sim \pi_{old}} \left[ \sum_t \frac{\pi_\theta(a_t | s_t)}{\pi_{old}(a_t | s_t)} \nabla_\theta \log \pi_\theta(a_t | s_t) \cdot \hat{A}_t \right]
$$

于是

$$\boxed{J(\theta) = \mathbb{E}_{\tau\sim\pi_{old}}\Big[\sum_t \underbrace{\frac{\pi_\theta(a_t|s_t)}{\pi_{old}(a_t|s_t)}}_{r_t(\theta)}\,\hat{A}_t\Big]}$$

定义 **$r_t(\theta) = \dfrac{\pi_\theta(a_t|s_t)}{\pi_{old}(a_t|s_t)}$** 为概率比（importance ratio）：
- $r_t = 1$：新旧策略对 $a_t$ 给的概率一样（策略没变）；
- $r_t > 1$：新策略**更可能**选 $a_t$；
- $r_t < 1$：新策略**更不可能**选 $a_t$。

> **结论：只要乘上重要性权重 $r_t(\theta)$，旧数据就能用于更新新策略——这就是 PPO 能"一批数据多用几次"的理论基础。**（A2C 不敢这么干，所以只能用一次。）

### 3.3 核心推导二：clip——为什么多次更新还安全？

#### 3.3.1 先看 TRPO 的做法（PPO 的前身）

TRPO 给目标加硬约束：

$$\max_\theta \mathbb{E}\big[r_t(\theta)\hat{A}_t\big] \quad \text{s.t.} \quad \text{KL}(\pi_{old}, \pi_\theta) \le \delta$$

限制新旧策略的 KL 距离不超 $\delta$。但这需要**二阶优化**（计算 KL 的 Hessian），实现复杂——这正是 ACKTR 那类方法的复杂度来源。

#### 3.3.2 PPO 的替代：把"约束"改成"clip 惩罚"

PPO 放弃硬约束，改用一行就能实现的 clip：

$$\boxed{L^{\text{CLIP}}(\theta) = \mathbb{E}_t\Big[\min\big(r_t(\theta)\hat{A}_t,\; \text{clip}(r_t(\theta),\,1-\varepsilon,\,1+\varepsilon)\,\hat{A}_t\big)\Big]}$$

$\varepsilon$ 通常取 0.2。**核心逻辑：只允许策略在概率比 $r$ 的 $[1-\varepsilon, 1+\varepsilon]$ 区间内自由优化，超出就截断。**

#### 3.3.3 为什么 min + clip 有效？（分两种情况推导）

**情况 A：$\hat{A}_t > 0$（好动作，想提高概率 → $r$ 增大）**

$$\min\big(r\hat{A},\;\text{clip}(r)\hat{A}\big) = \begin{cases} r\hat{A} & r \le 1+\varepsilon \\[4pt] (1+\varepsilon)\hat{A} & r > 1+\varepsilon \end{cases}$$

- $r$ 在 $[1, 1+\varepsilon]$ 内：正常优化，提高该动作概率（目标随之增大）；
- $r$ 超过 $1+\varepsilon$：clip 项变成常数 $(1+\varepsilon)\hat{A}$，$\min$ 取它 → **目标不再随 $r$ 增大，梯度为 0 → 单步最多把概率比推到 $1+\varepsilon$**。

**情况 B：$\hat{A}_t < 0$（坏动作，想降低概率 → $r$ 减小）**

$$\min\big(r\hat{A},\;\text{clip}(r)\hat{A}\big) = \begin{cases} r\hat{A} & r \ge 1-\varepsilon \\[4pt] (1-\varepsilon)\hat{A} & r < 1-\varepsilon \end{cases}$$

- $r$ 在 $[1-\varepsilon, 1]$ 内：正常优化，降低该动作概率；
- $r$ 低于 $1-\varepsilon$：clip 项 $(1-\varepsilon)\hat{A}$（负常数）**托底** → 即使继续压概率，目标也不再降低 → **防止一次坏运气就把动作概率压到 0（防止探索崩溃）**。

> **一句话总结 clip 的几何意义**：**好的更新有"上限"（不能太贪），坏的更新有"下限"（不能太狠）**——每一步更新都被夹在旧策略附近，这就是用一行 min + clip 实现的"信任区域"。

### 3.4 完整 PPO 损失与训练流程

$$L(\theta) = \underbrace{L^{\text{CLIP}}}_{\text{actor}} + c_1\underbrace{\big(V(s)-\text{return}\big)^2}_{\text{critic, 同 A2C}} - c_2\underbrace{H(\pi(\cdot|s))}_{\text{熵正则, 同 A2C}}$$

训练流程（与 A2C 几乎一样，只多一步"复用数据"）：

```
1. 用旧策略 π_old 采样一批数据（N 环境 × T 步）
   —— 额外存下每个动作的 log π_old(a|s)
2. 算 GAE 优势（沿用 2.6 节，λ≈0.95）
3. 对同一批数据做 K 个 epoch 的 mini-batch 更新：
   a. 重算 r_t(θ) = exp(log π_θ(a|s) - log π_old(a|s))
      （指数化：把"概率比"变成"对数概率差"，数值稳定）
   b. 套 clip 目标 → 更新
4. 数据用完即弃（仍是 on-policy，但每批高效复用 K 次）
```

### 3.5 PPO vs A2C 对照

| 维度 | A2C | PPO |
|---|---|---|
| 每批数据用几次 | 1 次（用完即弃） | K 次（如 4 epoch × mini-batch） |
| 更新步长保护 | 无（LR 难调，易震荡） | clip 夹住 $r \in [1\pm\varepsilon]$ |
| 信任区域 | 无 | clip 近似 TRPO 的 KL 约束 |
| critic / GAE / 熵 / 优势标准化 | 有 | **全部沿用** |
| LunarLander 体验 | 调参到怀疑人生 | 明显更稳、更易收敛 |

> **一句话**：PPO = "A2C 的全部组件 + 重要性采样 + clip"。critic 学习、GAE、熵、优势标准化、并行环境全部原封不动沿用，PPO 只多了概率比 $r_t(\theta)$ 和 clip 两样新东西。